In [1]:
# Fix sympy/torch conflict first, then install packages
!pip install -q sympy==1.13.1
!pip install -q transformers datasets seqeval accelerate sentencepiece protobuf
import importlib, sys
# Force reload sympy so the fixed version is active in this session
for mod in list(sys.modules.keys()):
    if 'sympy' in mod:
        del sys.modules[mod]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 69.7 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.13.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 86.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import ast, math, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from seqeval.metrics import (
    classification_report,
    f1_score as strict_f1_score,
    precision_score as strict_precision_score,
    recall_score as strict_recall_score,
)
from collections import Counter
from tqdm import tqdm

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # ── Reproducibility: make cuDNN deterministic ──────────────────────────────
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)


PyTorch: 2.10.0+cu128 | device: cuda


In [3]:
MODEL_NAME          = "microsoft/deberta-large"
MAX_LEN             = 256
BATCH_SIZE          = 4
ACCUMULATION_STEPS  = 8
NUM_EPOCHS          = 10
LR                  = 2e-5
WEIGHT_DECAY        = 0.01
WARMUP_RATIO        = 0.15
DROPOUT             = 0.2
EARLY_STOP_PATIENCE = 5
GRAD_CLIP           = 1.0
LABEL_SMOOTHING     = 0.05

SOCIAL_BOOST        = 1.35
BOUNDARY_WEIGHT     = 1.5

USE_ENTITY_AUGMENT  = True
AUG_COPIES          = 2

# ── 2 seeds for DeBERTa, 2 seeds for PubMedBERT ──────────────────────────────
DEBERTA_SEEDS   = [47, 49]        # best two from previous run
PUBMEDBERT_SEEDS = [41, 44]

SAVE_PATH  = "best_deberta_ner.pt"

TRAIN_PATH = "/kaggle/input/datasets/shuvodey001/smmh-sharedtask07/new_train_data.csv"
DEV_PATH   = "/kaggle/input/datasets/shuvodey001/smmh-sharedtask07/new_dev_data.csv"
TEST_PATH  = "/kaggle/input/datasets/shuvodey001/smmh-sharedtask07/new_test_data.csv"

# Keep SEEDS pointing at DeBERTa seeds so downstream cells that reference it still work
SEEDS = DEBERTA_SEEDS


In [4]:
label_list = ["O","B-ClinicalImpacts","I-ClinicalImpacts","B-SocialImpacts","I-SocialImpacts"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)
print("label2id:", label2id)


label2id: {'O': 0, 'B-ClinicalImpacts': 1, 'I-ClinicalImpacts': 2, 'B-SocialImpacts': 3, 'I-SocialImpacts': 4}


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [6]:
def parse_tokens(val) -> list:
    if isinstance(val, list):
        return [str(x).strip() for x in val if str(x).strip()]
    if pd.isna(val):
        return []
    text = str(val).strip()
    if not text:
        return []
    if text.startswith("["):
        try:
            return [str(x).strip() for x in ast.literal_eval(text) if str(x).strip()]
        except Exception:
            pass
    return [t for t in text.split() if t.strip()]


def parse_labels(val, n_tokens: int) -> list:
    # ── Guard: augmented rows pass a list directly — return it as-is ──────────
    if isinstance(val, list):
        raw_labels = [str(x).strip() for x in val]
        if len(raw_labels) < n_tokens:
            raw_labels += ["O"] * (n_tokens - len(raw_labels))
        raw_labels = raw_labels[:n_tokens]
        return [lbl if lbl in label2id else "O" for lbl in raw_labels]
    # ── Scalar path (CSV strings, NaN) ────────────────────────────────────────
    try:
        is_na = pd.isna(val)
    except (ValueError, TypeError):
        is_na = False
    if is_na:
        return ["O"] * n_tokens
    text = str(val).strip()
    if not text:
        return ["O"] * n_tokens
    if text.startswith("["):
        try:
            raw_labels = [str(x).strip() for x in ast.literal_eval(text)]
        except Exception:
            raw_labels = text.split()
    else:
        raw_labels = text.split()
    if len(raw_labels) < n_tokens:
        raw_labels += ["O"] * (n_tokens - len(raw_labels))
    raw_labels = raw_labels[:n_tokens]
    return [lbl if lbl in label2id else "O" for lbl in raw_labels]


In [7]:
train_raw = pd.read_csv(TRAIN_PATH)
dev_raw   = pd.read_csv(DEV_PATH)

# ── Combine train + dev for final submission training ─────────────────────────
train_pool  = train_raw.sample(frac=1, random_state=42).reset_index(drop=True)

entity_rows = train_pool[train_pool['ner_tags'].str.contains('B-', na=False)]
other_rows  = train_pool[~train_pool.index.isin(entity_rows.index)]
val_entity  = entity_rows.sample(frac=0.05, random_state=42)
val_other   = other_rows.sample(frac=0.05, random_state=42)
val_df      = pd.concat([val_entity, val_other]).sample(frac=1, random_state=42).reset_index(drop=True)
train_only  = train_pool[~train_pool.index.isin(val_df.index)].reset_index(drop=True)

# Merge original train (minus val slice) + full dev
train_df = pd.concat([train_only, dev_raw], ignore_index=True).sample(
    frac=1, random_state=42).reset_index(drop=True)

print(f"train_only: {len(train_only)} | dev: {len(dev_raw)} | combined train_df: {len(train_df)}")
print(f"Val (early-stop only): {len(val_df)} rows")

# ── Entity-replacement augmentation ──────────────────────────────────────────
CLINICAL_SYNONYMS = [
    "withdrawal", "depression", "anxiety", "hospitalized", "overdose",
    "relapse", "addiction", "rehab", "detox", "chronic pain",
    "insomnia", "nausea", "tremors", "psychosis", "suicidal thoughts",
    "went to rehab", "seeking help", "entered treatment", "mental breakdown",
    "panic attacks", "lost consciousness", "seizure", "in recovery",
]
SOCIAL_SYNONYMS = [
    "lost my job", "lost custody", "arrested", "homeless", "evicted",
    "divorced", "lost my apartment", "fired", "lost my license",
    "family cut me off", "went to jail", "lost my kids", "lost my home",
    "dropped out", "lost my friends", "relationship ended", "isolated",
    "financial ruin", "lost my car", "charged with possession",
]

def augment_row(tokens, labels):
    augmented = []
    spans = []
    i = 0
    while i < len(labels):
        if labels[i].startswith("B-"):
            etype = labels[i][2:]
            j = i + 1
            while j < len(labels) and labels[j] == f"I-{etype}":
                j += 1
            spans.append((i, j, etype))
            i = j
        else:
            i += 1
    if not spans:
        return augmented
    for _ in range(AUG_COPIES):
        new_tokens = tokens[:]
        new_labels = labels[:]
        offset = 0
        for (s, e, etype) in spans:
            pool = CLINICAL_SYNONYMS if etype == "ClinicalImpacts" else SOCIAL_SYNONYMS
            replacement = random.choice(pool).split()
            rs, re = s + offset, e + offset
            rep_labels = [f"B-{etype}"] + [f"I-{etype}"] * (len(replacement) - 1)
            new_tokens = new_tokens[:rs] + replacement + new_tokens[re:]
            new_labels = new_labels[:rs] + rep_labels  + new_labels[re:]
            offset += len(replacement) - (e - s)
        augmented.append((new_tokens, new_labels))
    return augmented

if USE_ENTITY_AUGMENT:
    aug_rows = []
    for _, row in train_df.iterrows():
        tokens = parse_tokens(row["tokens"])
        labels = parse_labels(row["ner_tags"], len(tokens))
        if any(l != "O" for l in labels):
            for aug_tok, aug_lbl in augment_row(tokens, labels):
                aug_rows.append({"tokens": aug_tok, "ner_tags": aug_lbl})
    if aug_rows:
        aug_df   = pd.DataFrame(aug_rows)
        train_df = pd.concat([train_df, aug_df], ignore_index=True).sample(
            frac=1, random_state=42).reset_index(drop=True)
        print(f"After augmentation: {len(train_df)} rows ({len(aug_rows)} new)")

# ── Boundary-aware loss ───────────────────────────────────────────────────────
import torch.nn as nn

# class_weights used by GenericNERModel loss in Cell 19
class_weights = torch.ones(NUM_LABELS)
for lbl, idx in label2id.items():
    if "Social" in lbl:
        class_weights[idx] *= SOCIAL_BOOST
class_weights = class_weights.to(device)

def build_loss_fn(label_smoothing: float = LABEL_SMOOTHING):
    base_weights = class_weights.clone()
    ce = nn.CrossEntropyLoss(
        weight=base_weights,
        ignore_index=-100,
        label_smoothing=label_smoothing,
    )
    def loss_fn(logits, labels):
        base_loss = ce(logits, labels)
        b_mask = torch.zeros_like(labels, dtype=torch.float)
        for lbl, idx in label2id.items():
            if lbl.startswith("B-"):
                b_mask = b_mask + (labels == idx).float()
        b_mask = b_mask * (labels != -100).float()
        if b_mask.sum() > 0:
            b_ce   = nn.CrossEntropyLoss(weight=base_weights, ignore_index=-100,
                                         label_smoothing=label_smoothing, reduction="none")
            b_loss = (b_ce(logits, labels) * b_mask).sum() / b_mask.sum()
            return base_loss + (BOUNDARY_WEIGHT - 1.0) * b_loss
        return base_loss
    return loss_fn

loss_fn = build_loss_fn()
print("Loss function built.")


train_only: 799 | dev: 258 | combined train_df: 1057
Val (early-stop only): 43 rows
After augmentation: 1695 rows (638 new)
Loss function built.


In [8]:
# ── ALWAYS run this cell before training and check the output ─────────────────
print("=" * 65)
print("RAW CSV VALUES (first row):")
print(f"  tokens col type : {type(train_df.iloc[0]['tokens']).__name__}")
print(f"  ner_tags col type : {type(train_df.iloc[0]['ner_tags']).__name__}")
print(f"  tokens raw      : {str(train_df.iloc[0]['tokens'])[:100]}")
print(f"  ner_tags raw    : {str(train_df.iloc[0]['ner_tags'])[:100]}")

sample_tokens = parse_tokens(train_df.iloc[0]["tokens"])
sample_labels = parse_labels(train_df.iloc[0]["ner_tags"], len(sample_tokens))
print()
print("PARSED (first row):")
print(f"  tokens : {sample_tokens[:15]}")
print(f"  labels : {sample_labels[:15]}")

# Show a row that actually has an entity label (not just all-O)
print()
print("Searching for a row with non-O labels...")
for i, row in train_df.iterrows():
    toks = parse_tokens(row["tokens"])
    lbls = parse_labels(row["ner_tags"], len(toks))
    if any(l != "O" for l in lbls):
        print(f"  Row {i} has entities:")
        for t, l in zip(toks[:20], lbls[:20]):
            marker = " <" if l != "O" else ""
            print(f"    {t:<20s} {l}{marker}")
        break
else:
    print("  WARNING: no rows with non-O entities found.")

# Full label distribution
print()
counter = Counter()
for _, row in train_df.iterrows():
    toks = parse_tokens(row["tokens"])
    lbls = parse_labels(row["ner_tags"], len(toks))
    counter.update(lbls)
total = sum(counter.values())
print("LABEL DISTRIBUTION (train):")
for lbl, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / total
    print(f"  {lbl:<25s}  {cnt:6d}  ({pct:5.1f}%)")
non_o = sum(v for k, v in counter.items() if k != "O")
print()
if non_o == 0:
    print("WARNING: ZERO non-O labels found. Check your CSV path/columns.")
else:
    print(f"Non-O entity tokens: {non_o} ({100*non_o/total:.1f}%)")


RAW CSV VALUES (first row):
  tokens col type : str
  ner_tags col type : str
  tokens raw      : ['I', 'almost', 'died', 'four', 'times', '.']
  ner_tags raw    : ['O', 'B-ClinicalImpacts', 'I-ClinicalImpacts', 'O', 'O', 'O']

PARSED (first row):
  tokens : ['I', 'almost', 'died', 'four', 'times', '.']
  labels : ['O', 'B-ClinicalImpacts', 'I-ClinicalImpacts', 'O', 'O', 'O']

Searching for a row with non-O labels...
  Row 0 has entities:
    I                    O
    almost               B-ClinicalImpacts <
    died                 I-ClinicalImpacts <
    four                 O
    times                O
    .                    O

LABEL DISTRIBUTION (train):
  O                           32364  ( 91.9%)
  B-ClinicalImpacts            1011  (  2.9%)
  I-ClinicalImpacts             769  (  2.2%)
  I-SocialImpacts               727  (  2.1%)
  B-SocialImpacts               330  (  0.9%)

Non-O entity tokens: 2837 (8.1%)


In [9]:
class NERDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.rows, self.skipped = [], 0
        for _, row in df.iterrows():
            raw_toks = row["tokens"]
            raw_lbls = row["ner_tags"]
            # Support both raw strings (CSV) and pre-parsed lists (augmented rows)
            if isinstance(raw_toks, list):
                tokens = [str(x).strip() for x in raw_toks if str(x).strip()]
                labels = raw_lbls if isinstance(raw_lbls, list) else parse_labels(raw_lbls, len(tokens))
            else:
                tokens = parse_tokens(raw_toks)
                labels = parse_labels(raw_lbls, len(tokens))
            if not tokens:
                self.skipped += 1
                continue
            self.rows.append({"tokens": tokens, "labels": labels})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        tokens = self.rows[idx]["tokens"]
        labels = self.rows[idx]["labels"]

        enc = tokenizer(
            tokens,
            is_split_into_words=True,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        word_ids = enc.word_ids()
        label_ids = []
        prev = None
        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                lbl = labels[w] if w < len(labels) else "O"
                label_ids.append(label2id.get(lbl, 0))
            else:
                label_ids.append(-100)
            prev = w

        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(label_ids, dtype=torch.long),
        }


In [10]:
train_dataset = NERDataset(train_df)
val_dataset   = NERDataset(val_df)
print(f"Train: {len(train_dataset)} rows (skipped {train_dataset.skipped})")
print(f"Val (early-stop only): {len(val_dataset)} rows (skipped {val_dataset.skipped})")

def _worker_init(worker_id):
    np.random.seed(42 + worker_id)
    random.seed(42 + worker_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=_worker_init,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=_worker_init,
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
first_batch = next(iter(train_loader))
non_o = (first_batch["labels"] > 0).sum().item()
print(f"First batch non-O positions: {non_o}")


Train: 1695 rows (skipped 0)
Val (early-stop only): 43 rows (skipped 0)
Train batches: 424 | Val batches: 11
First batch non-O positions: 10


In [11]:
class DebertaNER(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id,
            hidden_dropout_prob=DROPOUT,
            attention_probs_dropout_prob=DROPOUT,
            ignore_mismatched_sizes=True,
        )

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        return out


In [13]:
model = DebertaNER().to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

num_update_steps = math.ceil(len(train_loader) / ACCUMULATION_STEPS) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * num_update_steps),
    num_training_steps=num_update_steps,
)
print(f"Model params:          {sum(p.numel() for p in model.parameters()):,}")
print(f"Effective batch size:  {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"Total optimiser steps: {num_update_steps}")
print(f"Warmup steps:          {int(WARMUP_RATIO * num_update_steps)}")


pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [12]:
def extract_spans(label_seq):
    spans, start, cur = [], None, None
    for i, lbl in enumerate(label_seq):
        if lbl.startswith("B-"):
            if cur: spans.append((cur, start, i - 1))
            start, cur = i, lbl[2:]
        elif lbl.startswith("I-"):
            etype = lbl[2:]
            if cur != etype:
                if cur: spans.append((cur, start, i - 1))
                start, cur = i, etype
        else:
            if cur: spans.append((cur, start, i - 1))
            start, cur = None, None
    if cur: spans.append((cur, start, len(label_seq) - 1))
    return spans


def relaxed_f1(all_golds, all_preds):
    """Paper Section 3.5 relaxed token-level F1 — micro-avg overall."""
    tp_tok = Counter(); pred_tok = Counter(); gold_tok = Counter()
    for gold_seq, pred_seq in zip(all_golds, all_preds):
        g_spans = extract_spans(gold_seq)
        p_spans = extract_spans(pred_seq)
        all_types = set(t for t, *_ in g_spans + p_spans)
        for etype in all_types:
            g_set = [(s, e) for t, s, e in g_spans if t == etype]
            p_set = [(s, e) for t, s, e in p_spans if t == etype]
            for ps, pe in p_set:
                pred_tok[etype] += pe - ps + 1
                for gs, ge in g_set:
                    tp_tok[etype] += max(0, min(ge, pe) - max(gs, ps) + 1)
            for gs, ge in g_set:
                gold_tok[etype] += ge - gs + 1
    results = {}
    for etype in set(list(tp_tok) + list(gold_tok)):
        p = tp_tok[etype] / pred_tok[etype] if pred_tok[etype] else 0.0
        r = tp_tok[etype] / gold_tok[etype] if gold_tok[etype] else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        results[etype] = (round(p, 4), round(r, 4), round(f, 4))
    tp_all = sum(tp_tok.values()); pr_all = sum(pred_tok.values()); go_all = sum(gold_tok.values())
    op = tp_all / pr_all if pr_all else 0.0
    or_ = tp_all / go_all if go_all else 0.0
    of  = 2 * op * or_ / (op + or_) if (op + or_) else 0.0
    results["__all__"] = (round(op, 4), round(or_, 4), round(of, 4))
    return results


# ── Temperature-scaling calibration ───────────────────────────────────────────
# Learned on val set after training; divides logits to soften over-confident predictions.
TEMPERATURE = 1.0   # updated by calibrate_temperature() after each seed

def calibrate_temperature(model, loader, temps=None):
    """
    Grid-search temperature T in `temps` that maximises strict F1 on `loader`.
    Returns the best T. Sets global TEMPERATURE.
    """
    global TEMPERATURE
    if temps is None:
        temps = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.5]
    model.eval()
    # Collect raw logits and gold labels once
    all_logits, all_gold = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            amask = batch["attention_mask"].to(device)
            lbls  = batch["labels"]
            out   = model(input_ids=ids, attention_mask=amask)
            all_logits.append(out.logits.cpu())
            all_gold.append(lbls)

    best_t, best_f1 = 1.0, -1.0
    for T in temps:
        all_preds, all_golds = [], []
        for logits_batch, gold_batch in zip(all_logits, all_gold):
            pred_ids = (logits_batch / T).argmax(dim=-1)
            for pred_row, gold_row in zip(pred_ids, gold_batch):
                gold_lbls, pred_lbls = [], []
                for p, g in zip(pred_row.tolist(), gold_row.tolist()):
                    if g != -100:
                        gold_lbls.append(id2label[g])
                        pred_lbls.append(id2label[p])
                pred_lbls = enforce_bio_consistency(pred_lbls)
                if gold_lbls:
                    all_golds.append(gold_lbls)
                    all_preds.append(pred_lbls)
        f1 = strict_f1_score(all_golds, all_preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, T
    TEMPERATURE = best_t
    print(f"  Temperature calibrated: T={best_t:.2f}  (val strict F1={best_f1:.4f})")
    return best_t


In [13]:
# ── First-person filter ───────────────────────────────────────────────────────
# Zero out entity predictions for sentences with no first-person markers.
# Mirrors annotation guideline: only self-reported impacts are annotated.
FIRST_PERSON_MARKERS = {
    "i", "i'm", "i've", "i'd", "i'll", "im", "ive", "id",
    "my", "me", "myself", "mine", "we", "we're", "our", "ours",
}

def has_first_person(tokens):
    return any(t.lower() in FIRST_PERSON_MARKERS for t in tokens)


def apply_first_person_filter(tokens, pred_labels):
    """Replace all entity predictions with O when no first-person token found."""
    if not has_first_person(tokens):
        return ["O"] * len(pred_labels)
    return pred_labels


def enforce_bio_consistency(tags: list) -> list:
    fixed = []
    prev_type = None
    for t in tags:
        if t == "O":
            fixed.append("O"); prev_type = None; continue
        if t.startswith("B-"):
            ent = t[2:]; fixed.append(t); prev_type = ent; continue
        if t.startswith("I-"):
            ent = t[2:]
            fixed.append(t if prev_type == ent else f"B-{ent}")
            prev_type = ent; continue
        fixed.append("O"); prev_type = None
    return fixed


def evaluate(model, loader, verbose=True, ensemble_models=None, temperature=None):
    T = temperature if temperature is not None else TEMPERATURE
    models_to_use = ensemble_models if ensemble_models is not None else [model]
    for m in models_to_use:
        m.eval()
    all_preds, all_golds = [], []

    # We need tokens for the first-person filter; store them alongside the dataset rows
    dataset = loader.dataset
    row_tokens = [r["tokens"] for r in dataset.rows] if hasattr(dataset, "rows") else None
    row_idx = 0

    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            amask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            if len(models_to_use) > 1:
                logits_sum = None
                for m in models_to_use:
                    out = m(input_ids=ids, attention_mask=amask)
                    logits_sum = out.logits if logits_sum is None else logits_sum + out.logits
                pred_ids = (logits_sum / (T * len(models_to_use))).argmax(dim=-1).cpu()
            else:
                out = models_to_use[0](input_ids=ids, attention_mask=amask)
                pred_ids = (out.logits / T).argmax(dim=-1).cpu()

            for pred_row, gold_row in zip(pred_ids, labels):
                gold_lbls, pred_lbls = [], []
                for p, g in zip(pred_row.tolist(), gold_row.tolist()):
                    if g != -100:
                        gold_lbls.append(id2label[g])
                        pred_lbls.append(id2label[p])
                pred_lbls = enforce_bio_consistency(pred_lbls)
                # ── First-person filter ──────────────────────────────────────
                if row_tokens is not None and row_idx < len(row_tokens):
                    pred_lbls = apply_first_person_filter(row_tokens[row_idx], pred_lbls)
                if gold_lbls:
                    all_golds.append(gold_lbls)
                    all_preds.append(pred_lbls)
                row_idx += 1

    if not all_golds:
        print("No valid evaluation sequences.")
        return 0.0, {}

    s_f1 = strict_f1_score(all_golds, all_preds, zero_division=0)
    s_p  = strict_precision_score(all_golds, all_preds, zero_division=0)
    s_r  = strict_recall_score(all_golds, all_preds, zero_division=0)

    rel_metrics = relaxed_f1(all_golds, all_preds)
    relaxed_overall = rel_metrics.get("__all__", (0, 0, 0))[2]

    if verbose:
        print(f"\n  Strict  F1={s_f1:.4f}  P={s_p:.4f}  R={s_r:.4f}  (T={T:.2f})")
        print(f"  Relaxed F1={relaxed_overall:.4f}")
        for et in sorted(k for k in rel_metrics if k != "__all__"):
            p, r, f = rel_metrics[et]
            print(f"  {et:<25s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}")
        try:
            print("\n[Strict seqeval per-label report]")
            print(classification_report(all_golds, all_preds, zero_division=0))
        except Exception:
            pass

    return s_f1, rel_metrics


In [16]:
results = []

for seed in DEBERTA_SEEDS:
    set_seed(seed)
    seed_save_path = f"best_deberta_ner_seed{seed}.pt"
    print(f"\n{'='*65}")
    print(f"DeBERTa  seed={seed}  →  {seed_save_path}")
    print(f"{'='*65}")

    model = DebertaNER().to(device)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    num_update_steps = math.ceil(len(train_loader) / ACCUMULATION_STEPS) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * num_update_steps),
        num_training_steps=num_update_steps,
    )

    best_f1, patience_ctr, global_step = 0.0, 0, 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss, valid_steps, nan_steps = 0.0, 0, 0
        optimizer.zero_grad()
        bar = tqdm(train_loader, desc=f"[DeBERTa seed {seed}] Epoch {epoch+1}/{NUM_EPOCHS}")

        for step, batch in enumerate(bar):
            ids    = batch["input_ids"].to(device)
            amask  = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            out  = model(input_ids=ids, attention_mask=amask)
            loss = loss_fn(
                out.logits.view(-1, NUM_LABELS),
                labels.view(-1),
            ) / ACCUMULATION_STEPS

            if not torch.isfinite(loss):
                nan_steps += 1
                bar.set_postfix(loss="nan", nan=nan_steps)
                optimizer.zero_grad()
                continue

            loss.backward()

            is_update = (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader)
            if is_update:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            logged = loss.item() * ACCUMULATION_STEPS
            total_loss += logged
            valid_steps += 1
            bar.set_postfix(loss=f"{logged:.4f}", step=global_step)

        avg_loss = total_loss / max(valid_steps, 1)
        print(f"\n[DeBERTa seed {seed}] Epoch {epoch+1} Avg Loss: {avg_loss:.4f}  NaN: {nan_steps}")

        val_f1, _ = evaluate(model, val_loader, verbose=True, temperature=1.0)

        if val_f1 >= best_f1:
            best_f1, patience_ctr = val_f1, 0
            torch.save(model.state_dict(), seed_save_path)
            print(f"Best DeBERTa saved (Strict F1 = {best_f1:.4f})")
        else:
            patience_ctr += 1
            print(f"No improvement. Patience {patience_ctr}/{EARLY_STOP_PATIENCE}")
            if patience_ctr >= EARLY_STOP_PATIENCE:
                print("Early stopping triggered.")
                break

    results.append({"seed": seed, "path": seed_save_path, "best_f1": best_f1})
    print(f"\n[DeBERTa seed {seed}] Best Val Strict F1: {best_f1:.4f}")

    del model, optimizer, scheduler
    import gc
    gc.collect()
    torch.cuda.empty_cache()

print("\nDeBERTa training complete:")
for r in results:
    print(f"  seed={r['seed']}  F1={r['best_f1']:.4f}  path={r['path']}")



DeBERTa  seed=47  →  best_deberta_ner_seed47.pt


Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

DebertaForTokenClassification LOAD REPORT from: microsoft/deberta-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

[DeBERTa seed 47] Epoch 1/10: 100%|██████████| 424/424 [05:48<00:00,  1.21it/s, loss=0.3512, step=53]


[DeBERTa seed 47] Epoch 1 Avg Loss: 1.6585  NaN: 0



  Strict  F1=0.3913  P=0.3333  R=0.4737  (T=1.00)
  Relaxed F1=0.3636
  ClinicalImpacts            P=0.636  R=0.318  F1=0.424
  SocialImpacts              P=0.400  R=0.118  F1=0.182

[Strict seqeval per-label report]
                 precision    recall  f1-score   support

ClinicalImpacts       0.36      0.53      0.43        15
  SocialImpacts       0.20      0.25      0.22         4

      micro avg       0.33      0.47      0.39        19
      macro avg       0.28      0.39      0.33        19
   weighted avg       0.33      0.47      0.39        19

Best DeBERTa saved (Strict F1 = 0.3913)


...


[DeBERTa seed 49] Epoch 9 Avg Loss: 0.4076  NaN: 0



  Strict  F1=0.8205  P=0.8000  R=0.8421  (T=1.00)
  Relaxed F1=0.9748
  ClinicalImpacts            P=1.000  R=0.932  F1=0.965
  SocialImpacts              P=1.000  R=1.000  F1=1.000

[Strict seqeval per-label report]
                 precision    recall  f1-score   support

ClinicalImpacts       0.87      0.87      0.87        15
  SocialImpacts       0.60      0.75      0.67         4

      micro avg       0.80      0.84      0.82        19
      macro avg       0.73      0.81      0.77        19
   weighted avg       0.81      0.84      0.82        19

Best DeBERTa saved (Strict F1 = 0.8205)



[DeBERTa seed 49] Epoch 10/10: 100%|██████████| 424/424 [05:49<00:00,  1.21it/s, loss=0.3888, step=530]


[DeBERTa seed 49] Epoch 10 Avg Loss: 0.4007  NaN: 0



  Strict  F1=0.8205  P=0.8000  R=0.8421  (T=1.00)
  Relaxed F1=0.9748
  ClinicalImpacts            P=1.000  R=0.932  F1=0.965
  SocialImpacts              P=1.000  R=1.000  F1=1.000

[Strict seqeval per-label report]
                 precision    recall  f1-score   support

ClinicalImpacts       0.87      0.87      0.87        15
  SocialImpacts       0.60      0.75      0.67         4

      micro avg       0.80      0.84      0.82        19
      macro avg       0.73      0.81      0.77        19
   weighted avg       0.81      0.84      0.82        19

Best DeBERTa saved (Strict F1 = 0.8205)

[DeBERTa seed 49] Best Val Strict F1: 0.8205

DeBERTa training complete:
  seed=47  F1=0.8718  path=best_deberta_ner_seed47.pt
  seed=49  F1=0.8205  path=best_deberta_ner_seed49.pt


In [24]:
def build_tokenizer_for_model(model_name: str):
    # RoBERTa/DeBERTa tokenizers usually need add_prefix_space=True for token-level alignment.
    model_name_l = model_name.lower()
    kwargs = {"add_prefix_space": True} if ("roberta" in model_name_l or "deberta" in model_name_l) else {}
    return AutoTokenizer.from_pretrained(model_name, **kwargs)


In [20]:
# Compatibility definitions for running PubMedBERT cell before later utility cells
from transformers import AutoConfig

if "FlexibleNERDataset" not in globals():
    class FlexibleNERDataset(Dataset):
        def __init__(self, df: pd.DataFrame, tokenizer_obj, max_len: int):
            self.rows, self.skipped = [], 0
            self.tokenizer = tokenizer_obj
            self.max_len = max_len
            for _, row in df.iterrows():
                tokens = parse_tokens(row["tokens"])
                if not tokens:
                    self.skipped += 1
                    continue
                labels = parse_labels(row["ner_tags"], len(tokens))
                self.rows.append({"tokens": tokens, "labels": labels})

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, idx):
            tokens = self.rows[idx]["tokens"]
            labels = self.rows[idx]["labels"]
            enc = self.tokenizer(
                tokens,
                is_split_into_words=True,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt",
            )

            word_ids = enc.word_ids()
            label_ids = []
            prev = None
            for w in word_ids:
                if w is None:
                    label_ids.append(-100)
                elif w != prev:
                    lbl = labels[w] if w < len(labels) else "O"
                    label_ids.append(label2id.get(lbl, 0))
                else:
                    label_ids.append(-100)
                prev = w

            return {
                "input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels": torch.tensor(label_ids, dtype=torch.long),
            }

if "GenericNERModel" not in globals():
    class GenericNERModel(nn.Module):
        def __init__(self, model_name: str, dropout: float):
            super().__init__()
            config = AutoConfig.from_pretrained(
                model_name,
                num_labels=NUM_LABELS,
                id2label=id2label,
                label2id=label2id,
            )
            for attr in [
                "hidden_dropout_prob",
                "attention_probs_dropout_prob",
                "classifier_dropout",
                "dropout",
                "cls_dropout",
            ]:
                if hasattr(config, attr):
                    setattr(config, attr, dropout)
            self.model = AutoModelForTokenClassification.from_pretrained(
                model_name,
                config=config,
                ignore_mismatched_sizes=True,
            )

        def forward(self, input_ids, attention_mask, labels=None):
            return self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )


In [19]:
# ── PubMedBERT: 2-seed training ──────────────────────────────────────────────
import gc

PUBMEDBERT_MODEL  = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
PUBMEDBERT_PREFIX = "best_pubmedbert_ner"
PUBMEDBERT_MAX_LEN = 512   # BiomedBERT supports up to 512

pubmedbert_tokenizer = build_tokenizer_for_model(PUBMEDBERT_MODEL)

# Build dedicated loaders with PubMedBERT tokenizer
pubmed_train_dataset = FlexibleNERDataset(train_df, pubmedbert_tokenizer, PUBMEDBERT_MAX_LEN)
pubmed_val_dataset   = FlexibleNERDataset(val_df,   pubmedbert_tokenizer, PUBMEDBERT_MAX_LEN)

pubmed_train_loader = DataLoader(
    pubmed_train_dataset, batch_size=8, shuffle=True,
    num_workers=2, pin_memory=True,
)
pubmed_val_loader = DataLoader(
    pubmed_val_dataset, batch_size=8, shuffle=False,
    num_workers=2, pin_memory=True,
)
print(f"PubMedBERT train: {len(pubmed_train_dataset)} | val: {len(pubmed_val_dataset)}")

PUBMEDBERT_CFG = {
    "model_name":        PUBMEDBERT_MODEL,
    "display_name":      "PubMedBERT",
    "save_prefix":       PUBMEDBERT_PREFIX,
    "max_len":           PUBMEDBERT_MAX_LEN,
    "batch_size":        8,
    "accumulation_steps":4,
    "num_epochs":        10,
    "lr":                3e-5,
    "weight_decay":      0.01,
    "warmup_ratio":      0.1,
    "dropout":           0.1,
    "early_stop_patience": 5,
    "grad_clip":         1.0,
    "label_smoothing":   0.05,
}

pubmedbert_results = []

for seed in PUBMEDBERT_SEEDS:
    set_seed(seed)
    save_path = f"{PUBMEDBERT_PREFIX}_seed{seed}.pt"
    print(f"\n{'='*65}")
    print(f"PubMedBERT  seed={seed}  →  {save_path}")
    print(f"{'='*65}")

    pm_model = GenericNERModel(
        model_name=PUBMEDBERT_CFG["model_name"],
        dropout=PUBMEDBERT_CFG["dropout"],
    ).to(device)

    pm_optimizer = AdamW(pm_model.parameters(),
                         lr=PUBMEDBERT_CFG["lr"],
                         weight_decay=PUBMEDBERT_CFG["weight_decay"])

    num_update_steps = math.ceil(
        len(pubmed_train_loader) / PUBMEDBERT_CFG["accumulation_steps"]
    ) * PUBMEDBERT_CFG["num_epochs"]

    pm_scheduler = get_linear_schedule_with_warmup(
        pm_optimizer,
        num_warmup_steps=int(PUBMEDBERT_CFG["warmup_ratio"] * num_update_steps),
        num_training_steps=num_update_steps,
    )

    pm_loss_fn = build_loss_fn(PUBMEDBERT_CFG["label_smoothing"])

    print(f"Model params:          {sum(p.numel() for p in pm_model.parameters()):,}")
    print(f"Effective batch size:  {PUBMEDBERT_CFG['batch_size'] * PUBMEDBERT_CFG['accumulation_steps']}")
    print(f"Total optimiser steps: {num_update_steps}")

    best_f1, patience_ctr, global_step = 0.0, 0, 0

    for epoch in range(PUBMEDBERT_CFG["num_epochs"]):
        pm_model.train()
        total_loss, valid_steps, nan_steps = 0.0, 0, 0
        pm_optimizer.zero_grad()
        bar = tqdm(pubmed_train_loader, desc=f"[PubMedBERT seed {seed}] Epoch {epoch+1}/{PUBMEDBERT_CFG['num_epochs']}")

        for step, batch in enumerate(bar):
            ids    = batch["input_ids"].to(device)
            amask  = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            out  = pm_model(input_ids=ids, attention_mask=amask)
            loss = pm_loss_fn(
                out.logits.view(-1, NUM_LABELS),
                labels.view(-1),
            ) / PUBMEDBERT_CFG["accumulation_steps"]

            if not torch.isfinite(loss):
                nan_steps += 1
                pm_optimizer.zero_grad()
                continue

            loss.backward()

            is_update = ((step + 1) % PUBMEDBERT_CFG["accumulation_steps"] == 0
                         or (step + 1) == len(pubmed_train_loader))
            if is_update:
                torch.nn.utils.clip_grad_norm_(pm_model.parameters(),
                                               max_norm=PUBMEDBERT_CFG["grad_clip"])
                pm_optimizer.step()
                pm_scheduler.step()
                pm_optimizer.zero_grad()
                global_step += 1

            logged = loss.item() * PUBMEDBERT_CFG["accumulation_steps"]
            total_loss += logged
            valid_steps += 1
            bar.set_postfix(loss=f"{logged:.4f}", step=global_step)

        avg_loss = total_loss / max(valid_steps, 1)
        print(f"\n[PubMedBERT seed {seed}] Epoch {epoch+1} Avg Loss: {avg_loss:.4f}  NaN: {nan_steps}")

        # Evaluate using the generic evaluate() — pass pm_model directly
        val_f1, _ = evaluate(pm_model, pubmed_val_loader, verbose=True, temperature=1.0)

        if val_f1 >= best_f1:
            best_f1, patience_ctr = val_f1, 0
            torch.save(pm_model.state_dict(), save_path)
            print(f"Best PubMedBERT saved (Strict F1 = {best_f1:.4f})")
        else:
            patience_ctr += 1
            print(f"No improvement. Patience {patience_ctr}/{PUBMEDBERT_CFG['early_stop_patience']}")
            if patience_ctr >= PUBMEDBERT_CFG["early_stop_patience"]:
                print("Early stopping triggered.")
                break

    pubmedbert_results.append({"seed": seed, "path": save_path, "best_f1": best_f1})
    print(f"\n[PubMedBERT seed {seed}] Best Val Strict F1: {best_f1:.4f}")

    del pm_model, pm_optimizer, pm_scheduler
    gc.collect()
    torch.cuda.empty_cache()

print("\nPubMedBERT training complete:")
for r in pubmedbert_results:
    print(f"  seed={r['seed']}  F1={r['best_f1']:.4f}  path={r['path']}")


PubMedBERT train: 1695 | val: 43

PubMedBERT  seed=41  →  best_pubmedbert_ner_seed41.pt


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...


[PubMedBERT seed 44] Epoch 9 Avg Loss: 0.4289  NaN: 0



  Strict  F1=0.6341  P=0.5909  R=0.6842  (T=1.00)
  Relaxed F1=0.9043
  ClinicalImpacts            P=0.972  R=0.795  F1=0.875
  SocialImpacts              P=0.944  R=1.000  F1=0.971

[Strict seqeval per-label report]
                 precision    recall  f1-score   support

ClinicalImpacts       0.62      0.67      0.65        15
  SocialImpacts       0.50      0.75      0.60         4

      micro avg       0.59      0.68      0.63        19
      macro avg       0.56      0.71      0.62        19
   weighted avg       0.60      0.68      0.64        19

Best PubMedBERT saved (Strict F1 = 0.6341)


[PubMedBERT seed 44] Epoch 10/10: 100%|██████████| 212/212 [02:36<00:00,  1.35it/s, loss=0.4179, step=530]


[PubMedBERT seed 44] Epoch 10 Avg Loss: 0.4210  NaN: 0



  Strict  F1=0.5854  P=0.5455  R=0.6316  (T=1.00)
  Relaxed F1=0.8966
  ClinicalImpacts            P=0.972  R=0.795  F1=0.875
  SocialImpacts              P=0.895  R=1.000  F1=0.944

[Strict seqeval per-label report]
                 precision    recall  f1-score   support

ClinicalImpacts       0.62      0.67      0.65        15
  SocialImpacts       0.33      0.50      0.40         4

      micro avg       0.55      0.63      0.59        19
      macro avg       0.48      0.58      0.52        19
   weighted avg       0.56      0.63      0.59        19

No improvement. Patience 1/5

[PubMedBERT seed 44] Best Val Strict F1: 0.6341

PubMedBERT training complete:
  seed=41  F1=0.6829  path=best_pubmedbert_ner_seed41.pt
  seed=44  F1=0.6341  path=best_pubmedbert_ner_seed44.pt


In [ ]:


import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# def load_best_model(path: str = SAVE_PATH) -> DebertaNER:
#     m = DebertaNER().to(device)
#     m.load_state_dict(torch.load(path, map_location=device))
#     m.eval()
#     print(f"Loaded: {path}")
#     return m

# def load_ensemble_models(paths: list) -> list:
#     return [load_best_model(p) for p in paths]

# @torch.no_grad()
# def predict(model: DebertaNER, tokens: list) -> list:
#     enc = tokenizer(
#         tokens,
#         is_split_into_words=True,
#         padding="max_length",
#         truncation=True,
#         max_length=MAX_LEN,
#         return_tensors="pt",
#     )
#     out = model(
#         input_ids=enc["input_ids"].to(device),
#         attention_mask=enc["attention_mask"].to(device),
#     )
#     pred_ids = out.logits.argmax(dim=-1).squeeze(0).tolist()
#     word_ids = enc.word_ids()
#     final, prev = [], None
#     for idx, w in enumerate(word_ids):
#         if w is not None and w != prev:
#             final.append(id2label[pred_ids[idx]])
#         prev = w
#     return final

# @torch.no_grad()
# def predict_ensemble(models: list, tokens: list) -> list:
#     enc = tokenizer(
#         tokens,
#         is_split_into_words=True,
#         padding="max_length",
#         truncation=True,
#         max_length=MAX_LEN,
#         return_tensors="pt",
#     )
#     logits_sum = None
#     input_ids = enc["input_ids"].to(device)
#     attention_mask = enc["attention_mask"].to(device)
#     for m in models:
#         out = m(input_ids=input_ids, attention_mask=attention_mask)
#         logits_sum = out.logits if logits_sum is None else logits_sum + out.logits
#     pred_ids = (logits_sum / len(models)).argmax(dim=-1).squeeze(0).tolist()
#     word_ids = enc.word_ids()
#     final, prev = [], None
#     for idx, w in enumerate(word_ids):
#         if w is not None and w != prev:
#             final.append(id2label[pred_ids[idx]])
#         prev = w
#     return final

# # ── Ensemble evaluation ───────────────────────────────────────────────────────
# # Pick top-K checkpoints by val F1 from the multi-seed run above
# checkpoint_paths = [r["path"] for r in sorted(results, key=lambda x: -x["best_f1"])[:TOP_K_ENSEMBLE]]
# print("Ensemble checkpoints:", checkpoint_paths)

# ensemble_models = load_ensemble_models(checkpoint_paths)
# ensemble_f1, _ = evaluate(ensemble_models[0], val_loader, verbose=True, ensemble_models=ensemble_models)
# print(f"\nEnsemble Val Relaxed F1: {ensemble_f1:.4f}")

# # ── Sample token-level prediction ─────────────────────────────────────────────
# sample = ["I", "lost", "my", "job", "and", "felt", "deeply", "depressed", "for", "weeks"]
# preds = predict_ensemble(ensemble_models, sample)
# print("\nSample prediction:")
# print(f"  {'Token':<15}  Label")
# print(f"  {'-'*38}")
# for tok, lbl in zip(sample, preds):
#     print(f"  {tok:<15}  {lbl}{'  <' if lbl != 'O' else ''}")
def load_best_model(path: str = SAVE_PATH) -> DebertaNER:
    m = DebertaNER().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    print(f"Loaded: {path}")
    return m

def load_ensemble_models(paths: list) -> list:
    return [load_best_model(p) for p in paths]

@torch.no_grad()
def predict(model: DebertaNER, tokens: list) -> list:
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    out = model(
        input_ids=enc["input_ids"].to(device),
        attention_mask=enc["attention_mask"].to(device),
    )
    pred_ids = out.logits.argmax(dim=-1).squeeze(0).tolist()
    word_ids = enc.word_ids()
    final, prev = [], None
    for idx, w in enumerate(word_ids):
        if w is not None and w != prev:
            final.append(id2label[pred_ids[idx]])
        prev = w
    return final

@torch.no_grad()
def predict_ensemble(models: list, tokens: list) -> list:
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    # Accumulate logits on CPU across sequential model loads
    logits_sum = None
    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    for path in models:  # `models` is now a list of paths, not loaded models
        m = load_best_model(path)
        out = m(
            input_ids=input_ids.to(device),
            attention_mask=attention_mask.to(device),
        )
        logits_cpu = out.logits.cpu()  # move to CPU immediately
        logits_sum = logits_cpu if logits_sum is None else logits_sum + logits_cpu

        # Unload immediately
        m.to("cpu")
        del m
        gc.collect()
        torch.cuda.empty_cache()

    pred_ids = (logits_sum / len(models)).argmax(dim=-1).squeeze(0).tolist()
    word_ids = enc.word_ids()
    final, prev = [], None
    for idx, w in enumerate(word_ids):
        if w is not None and w != prev:
            final.append(id2label[pred_ids[idx]])
        prev = w
    return final

# ── Ensemble evaluation ───────────────────────────────────────────────────────
# Pick top-K checkpoints by val F1 from the multi-seed run above
checkpoint_paths = [r["path"] for r in sorted(results, key=lambda x: -x["best_f1"])[:TOP_K_ENSEMBLE]]
print("Ensemble checkpoints:", checkpoint_paths)

# Pass paths instead of loaded models — sequential load/unload inside predict_ensemble
ensemble_models = checkpoint_paths  # no longer pre-loaded

# For evaluate(), load one model at a time and accumulate
print("\nRunning ensemble evaluation sequentially...")
all_logits = []
for path in checkpoint_paths:
    m = load_best_model(path)
    ensemble_f1, _ = evaluate(m, val_loader, verbose=False, ensemble_models=None)
    all_logits.append(ensemble_f1)
    m.to("cpu")
    del m
    gc.collect()
    torch.cuda.empty_cache()

ensemble_f1 = sum(all_logits) / len(all_logits)
print(f"\nEnsemble Val Relaxed F1: {ensemble_f1:.4f}")

# ── Sample token-level prediction ─────────────────────────────────────────────
sample = ["I", "lost", "my", "job", "and", "felt", "deeply", "depressed", "for", "weeks"]
preds = predict_ensemble(checkpoint_paths, sample)  # paths, not models
print("\nSample prediction:")
print(f"  {'Token':<15}  Label")
print(f"  {'-'*38}")
for tok, lbl in zip(sample, preds):
    print(f"  {tok:<15}  {lbl}{'  <' if lbl != 'O' else ''}")

In [ ]:
from transformers import AutoConfig

# PubMedBERT removed as requested. Keep this list empty.
ADDITIONAL_MODEL_CONFIGS = []


def build_tokenizer_for_model(model_name: str):
    try:
        return AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
    except TypeError:
        return AutoTokenizer.from_pretrained(model_name)


class FlexibleNERDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer_obj, max_len: int):
        self.rows, self.skipped = [], 0
        self.tokenizer = tokenizer_obj
        self.max_len = max_len
        for _, row in df.iterrows():
            tokens = parse_tokens(row["tokens"])
            if not tokens:
                self.skipped += 1
                continue
            labels = parse_labels(row["ner_tags"], len(tokens))
            self.rows.append({"tokens": tokens, "labels": labels})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        tokens = self.rows[idx]["tokens"]
        labels = self.rows[idx]["labels"]
        enc = self.tokenizer(
            tokens,
            is_split_into_words=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        word_ids = enc.word_ids()
        label_ids = []
        prev = None
        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                lbl = labels[w] if w < len(labels) else "O"
                label_ids.append(label2id.get(lbl, 0))
            else:
                label_ids.append(-100)
            prev = w

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(label_ids, dtype=torch.long),
        }


class GenericNERModel(nn.Module):
    def __init__(self, model_name: str, dropout: float):
        super().__init__()
        config = AutoConfig.from_pretrained(
            model_name,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id,
        )
        for attr in [
            "hidden_dropout_prob",
            "attention_probs_dropout_prob",
            "classifier_dropout",
            "dropout",
            "cls_dropout",
        ]:
            if hasattr(config, attr):
                setattr(config, attr, dropout)
        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            config=config,
            ignore_mismatched_sizes=True,
        )

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )


def build_model_loaders(model_cfg: dict):
    model_tokenizer = build_tokenizer_for_model(model_cfg["model_name"])
    train_dataset_local = FlexibleNERDataset(train_df, model_tokenizer, model_cfg["max_len"])
    val_dataset_local = FlexibleNERDataset(val_df, model_tokenizer, model_cfg["max_len"])

    train_loader_local = DataLoader(
        train_dataset_local,
        batch_size=model_cfg["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader_local = DataLoader(
        val_dataset_local,
        batch_size=model_cfg["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    print(f"\n[{model_cfg['display_name']}] Train rows: {len(train_dataset_local)} | Val rows: {len(val_dataset_local)}")
    print(f"[{model_cfg['display_name']}] Train batches: {len(train_loader_local)} | Val batches: {len(val_loader_local)}")
    return model_tokenizer, train_loader_local, val_loader_local


def build_loss_fn(label_smoothing: float):
    return nn.CrossEntropyLoss(
        weight=class_weights,
        ignore_index=-100,
        label_smoothing=label_smoothing,
    )

In [ ]:
def train_additional_model(model_cfg: dict, seeds=None):
    seeds = seeds or SEEDS
    _, train_loader_local, val_loader_local = build_model_loaders(model_cfg)

    seed_runs = []
    for seed in seeds:
        set_seed(seed)
        save_path = f"{model_cfg['save_prefix']}_seed{seed}.pt"
        print(f"\n{'='*75}")
        print(f"Training {model_cfg['display_name']} | seed={seed} -> {save_path}")
        print(f"{'='*75}")

        model = GenericNERModel(
            model_name=model_cfg["model_name"],
            dropout=model_cfg["dropout"],
        ).to(device)
        optimizer = AdamW(
            model.parameters(),
            lr=model_cfg["lr"],
            weight_decay=model_cfg["weight_decay"],
        )
        num_update_steps = math.ceil(len(train_loader_local) / model_cfg["accumulation_steps"]) * model_cfg["num_epochs"]
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(model_cfg["warmup_ratio"] * num_update_steps),
            num_training_steps=num_update_steps,
        )
        loss_fn_local = build_loss_fn(model_cfg["label_smoothing"])

        print(f"Model params:          {sum(p.numel() for p in model.parameters()):,}")
        print(f"Effective batch size:  {model_cfg['batch_size'] * model_cfg['accumulation_steps']}")
        print(f"Total optimiser steps: {num_update_steps}")
        print(f"Warmup steps:          {int(model_cfg['warmup_ratio'] * num_update_steps)}")

        best_f1, patience_ctr, global_step = 0.0, 0, 0

        for epoch in range(model_cfg["num_epochs"]):
            model.train()
            total_loss, valid_steps, nan_steps = 0.0, 0, 0
            optimizer.zero_grad()
            bar = tqdm(train_loader_local, desc=f"[{model_cfg['display_name']}] seed {seed} epoch {epoch+1}/{model_cfg['num_epochs']}")

            for step, batch in enumerate(bar):
                ids = batch["input_ids"].to(device)
                amask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                out = model(input_ids=ids, attention_mask=amask)
                loss = loss_fn_local(
                    out.logits.view(-1, NUM_LABELS),
                    labels.view(-1),
                ) / model_cfg["accumulation_steps"]

                if not torch.isfinite(loss):
                    nan_steps += 1
                    bar.set_postfix(loss="nan", nan=nan_steps)
                    optimizer.zero_grad()
                    continue

                loss.backward()

                is_update = (step + 1) % model_cfg["accumulation_steps"] == 0 or (step + 1) == len(train_loader_local)
                if is_update:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=model_cfg["grad_clip"])
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                logged = loss.item() * model_cfg["accumulation_steps"]
                total_loss += logged
                valid_steps += 1
                bar.set_postfix(loss=f"{logged:.4f}", step=global_step)

            avg_loss = total_loss / max(valid_steps, 1)
            print(
                f"\n[{model_cfg['display_name']}] seed {seed} epoch {epoch+1}/{model_cfg['num_epochs']} "
                f"Avg Loss: {avg_loss:.4f}  NaN: {nan_steps}"
            )

            val_f1, _ = evaluate(model, val_loader_local, verbose=True)
            if val_f1 >= best_f1:
                best_f1, patience_ctr = val_f1, 0
                torch.save(model.state_dict(), save_path)
                print(f"Best model saved (Relaxed F1 = {best_f1:.4f})")
            else:
                patience_ctr += 1
                print(f"No improvement. Patience {patience_ctr}/{model_cfg['early_stop_patience']}")
                if patience_ctr >= model_cfg["early_stop_patience"]:
                    print("Early stopping triggered.")
                    break

        print(f"\n[{model_cfg['display_name']}] seed {seed} best Val Relaxed F1: {best_f1:.4f}")
        seed_runs.append({"seed": seed, "best_f1": best_f1, "path": save_path})

    best_run = max(seed_runs, key=lambda x: x["best_f1"])
    return {
        "display_name": model_cfg["display_name"],
        "model_name": model_cfg["model_name"],
        "hyperparameters": model_cfg,
        "seed_runs": seed_runs,
        "best_run": best_run,
        "mean_seed_f1": float(np.mean([x["best_f1"] for x in seed_runs])),
    }


additional_model_runs = {}
for model_cfg in ADDITIONAL_MODEL_CONFIGS:
    additional_model_runs[model_cfg["display_name"]] = train_additional_model(model_cfg)

print("\nFinished training additional models:")
for model_name, run_info in additional_model_runs.items():
    print(model_name, "->", run_info["best_run"])

In [ ]:
from IPython.display import display

comparison_rows = []
additional_runs = globals().get("additional_model_runs", {})

if "results" in globals() and results:
    deberta_best = max(results, key=lambda x: x["best_f1"])
    comparison_rows.append({
        "model": "DeBERTa Large",
        "model_id": MODEL_NAME,
        "best_val_f1": round(deberta_best["best_f1"], 4),
        "mean_seed_f1": round(float(np.mean([r["best_f1"] for r in results])), 4),
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "effective_batch": BATCH_SIZE * ACCUMULATION_STEPS,
        "max_len": MAX_LEN,
        "num_epochs": NUM_EPOCHS,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "dropout": DROPOUT,
        "best_checkpoint": deberta_best["path"],
    })

for model_cfg in ADDITIONAL_MODEL_CONFIGS:
    run_info = additional_runs.get(model_cfg["display_name"])
    if not run_info:
        continue
    comparison_rows.append({
        "model": model_cfg["display_name"],
        "model_id": model_cfg["model_name"],
        "best_val_f1": round(run_info["best_run"]["best_f1"], 4),
        "mean_seed_f1": round(run_info["mean_seed_f1"], 4),
        "batch_size": model_cfg["batch_size"],
        "accumulation_steps": model_cfg["accumulation_steps"],
        "effective_batch": model_cfg["batch_size"] * model_cfg["accumulation_steps"],
        "max_len": model_cfg["max_len"],
        "num_epochs": model_cfg["num_epochs"],
        "lr": model_cfg["lr"],
        "warmup_ratio": model_cfg["warmup_ratio"],
        "dropout": model_cfg["dropout"],
        "best_checkpoint": run_info["best_run"]["path"],
    })

comparison_df = pd.DataFrame(comparison_rows)
if comparison_df.empty:
    print("No comparison rows yet.")
    print("Run the DeBERTa training cell and the additional-model training cell first, then rerun this cell.")
else:
    comparison_df = comparison_df.sort_values("best_val_f1", ascending=False).reset_index(drop=True)
    print("Model comparison table:")
    display(comparison_df)
    try:
        print("\nMarkdown view:\n")
        print(comparison_df.to_markdown(index=False))
    except Exception:
        pass

In [ ]:
import ast
from typing import List, NamedTuple
from collections import defaultdict
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report


def _to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        return ast.literal_eval(x)
    raise TypeError(f"Unsupported cell type: {type(x)}")


def evaluate_test_strict_ner(
    df: pd.DataFrame,
    gold_col: str = "test",
    pred_col: str = "prediction",
    print_report: bool = True,
):
    y_true = df[gold_col].apply(_to_list).tolist()
    y_pred = df[pred_col].apply(_to_list).tolist()

    if len(y_true) != len(y_pred):
        raise ValueError(f"Row mismatch: gold rows={len(y_true)} pred rows={len(y_pred)}")

    for i, (g, p) in enumerate(zip(y_true, y_pred)):
        if len(g) != len(p):
            raise ValueError(
                f"Token length mismatch at row {i}: gold={len(g)} pred={len(p)}\n"
                f"gold: {g}\n"
                f"pred: {p}"
            )

    metrics = {
        "precision_strict": precision_score(y_true, y_pred),
        "recall_strict": recall_score(y_true, y_pred),
        "f1_strict": f1_score(y_true, y_pred),
        "accuracy_strict": accuracy_score(y_true, y_pred),
    }

    if print_report:
        print("=== STRICT ENTITY NER METRICS (seqeval) ===")
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")
        print("\n=== PER-LABEL REPORT ===")
        print(classification_report(y_true, y_pred, digits=4))

    return metrics


class Entity(NamedTuple):
    e_type: str
    start_offset: int
    end_offset: int


def bio_to_entities(bio_tags: List[str]) -> List[Entity]:
    entities = []
    start = None
    entity_type = None

    for i, tag in enumerate(bio_tags):
        if tag.startswith("B-"):
            if entity_type is not None:
                entities.append(Entity(e_type=entity_type, start_offset=start, end_offset=i - 1))
            entity_type = tag[2:]
            start = i
        elif tag.startswith("I-") and entity_type == tag[2:]:
            continue
        elif tag.startswith("I-") and entity_type != tag[2:]:
            if entity_type is not None:
                entities.append(Entity(e_type=entity_type, start_offset=start, end_offset=i - 1))
            entity_type = tag[2:]
            start = i
        elif tag == "O":
            if entity_type is not None:
                entities.append(Entity(e_type=entity_type, start_offset=start, end_offset=i - 1))
                entity_type = None
                start = None

    if entity_type is not None:
        entities.append(Entity(e_type=entity_type, start_offset=start, end_offset=len(bio_tags) - 1))

    return entities


def relaxed_overlap(entity1: Entity, entity2: Entity) -> float:
    if entity1.e_type != entity2.e_type:
        return 0
    return max(0, min(entity1.end_offset, entity2.end_offset) - max(entity1.start_offset, entity2.start_offset) + 1)


def calculate_f1_per_entity_covering_all(gold_labels: List[List[str]], pred_labels: List[List[str]]) -> dict:
    aggregated_results = defaultdict(lambda: {"TP_overlap": 0, "Total_True_Length": 0, "Total_Pred_Length": 0})

    for gold, pred in zip(gold_labels, pred_labels):
        true_entities = bio_to_entities(gold)
        pred_entities = bio_to_entities(pred)

        matched_pred_indices = set()
        for true_entity in true_entities:
            for i, pred_entity in enumerate(pred_entities):
                if i in matched_pred_indices:
                    continue
                overlap = relaxed_overlap(true_entity, pred_entity)
                if overlap > 0:
                    aggregated_results[true_entity.e_type]["TP_overlap"] += overlap
                    matched_pred_indices.add(i)
            aggregated_results[true_entity.e_type]["Total_True_Length"] += (true_entity.end_offset - true_entity.start_offset + 1)

        for pred_entity in pred_entities:
            aggregated_results[pred_entity.e_type]["Total_Pred_Length"] += (pred_entity.end_offset - pred_entity.start_offset + 1)

    final_results = {}
    overall_tp_overlap = 0
    overall_true_length = 0
    overall_pred_length = 0

    for entity_type, values in aggregated_results.items():
        precision = values["TP_overlap"] / values["Total_Pred_Length"] if values["Total_Pred_Length"] > 0 else 0.0
        recall = values["TP_overlap"] / values["Total_True_Length"] if values["Total_True_Length"] > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        overall_tp_overlap += values["TP_overlap"]
        overall_true_length += values["Total_True_Length"]
        overall_pred_length += values["Total_Pred_Length"]

        final_results[entity_type] = {
            "Precision": round(precision, 3),
            "Recall": round(recall, 3),
            "F1-Score": round(f1, 3),
            "Coverage": f"{values['TP_overlap']}/{values['Total_True_Length']}"
        }

    overall_precision = overall_tp_overlap / overall_pred_length if overall_pred_length > 0 else 0.0
    overall_recall = overall_tp_overlap / overall_true_length if overall_true_length > 0 else 0.0
    overall_f1 = (2 * overall_precision * overall_recall) / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0.0

    final_results["Overall"] = {
        "Precision": round(overall_precision, 3),
        "Recall": round(overall_recall, 3),
        "F1-Score": round(overall_f1, 3),
        "Coverage": f"{overall_tp_overlap}/{overall_true_length}"
    }
    return final_results

In [ ]:
# DEV data is merged into training. Val is a 5% early-stop slice from train only.
print("DEV merged into training. Both DeBERTa and PubMedBERT models trained.")


In [ ]:
import os

seeds = [41, 44, 45, 46, 47, 49]
for s in seeds:
    path = f"/kaggle/working/best_deberta_ner_seed{s}.pt"
    print(f"Seed {s}: {'✅ Found' if os.path.exists(path) else '❌ Missing'} — {path}")

In [ ]:
import os

SEEDS = [41, 44, 45, 46, 47, 49]

seed_to_path = {
    s: f"/kaggle/working/best_deberta_ner_seed{s}.pt"
    for s in SEEDS
}

seed_to_f1 = {s: 1.0 for s in SEEDS}  # dummy f1, adjust if your code uses it

print("seed_to_path ready:")
for s, p in seed_to_path.items():
    print(f"  Seed {s}: {p} — {'✅' if os.path.exists(p) else '❌'}")

In [12]:
import os

SEEDS = [ 47, 49]

# Reconstruct seed_to_path and seed_to_f1
seed_to_path = {s: f"/kaggle/working/best_deberta_ner_seed{s}.pt" for s in SEEDS}
seed_to_f1   = {s: 1.0 for s in SEEDS}

# Reconstruct `results` list that Cell 16 depends on
results = [
    {"seed": s, "path": seed_to_path[s], "best_f1": 1.0}
    for s in SEEDS
    if os.path.exists(seed_to_path[s])
]

# Reconstruct checkpoint_paths that ensemble uses
checkpoint_paths = [r["path"] for r in results]

print(f"Reconstructed {len(results)} checkpoints:")
for r in results:
    print(f"  Seed {r['seed']}: {r['path']} ✅")

Reconstructed 2 checkpoints:
  Seed 47: /kaggle/working/best_deberta_ner_seed47.pt ✅
  Seed 49: /kaggle/working/best_deberta_ner_seed49.pt ✅


In [22]:
import os

# ── Reconstruct PubMedBERT checkpoint registry ────────────────────────────────
seed_to_path = {s: f"/kaggle/working/best_pubmedbert_ner_seed{s}.pt" for s in PUBMEDBERT_SEEDS}
seed_to_f1   = {s: 1.0 for s in PUBMEDBERT_SEEDS}

pubmedbert_results = [
    {"seed": s, "path": seed_to_path[s], "best_f1": 1.0}
    for s in PUBMEDBERT_SEEDS
    if os.path.exists(seed_to_path[s])
]

print(f"Reconstructed {len(pubmedbert_results)} PubMedBERT checkpoints:")
for r in pubmedbert_results:
    print(f"  Seed {r['seed']}: {r['path']} ✅")

Reconstructed 2 PubMedBERT checkpoints:
  Seed 41: /kaggle/working/best_pubmedbert_ner_seed41.pt ✅
  Seed 44: /kaggle/working/best_pubmedbert_ner_seed44.pt ✅


In [ ]:
import torch, gc, os

# ── Step 1: Evaluate each checkpoint one at a time, never keep >1 in memory ──
seed_scores = {}

for s in SEEDS:
    path = seed_to_path[s]
    print(f"\nEvaluating seed {s}...")

    # Load single model
    m = DebertaNER().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()

    # Evaluate (use your existing evaluate() function)
    f1 = evaluate(m, val_loader, verbose=False)   # returns a float F1
    seed_scores[s] = f1
    print(f"  Seed {s} → F1: {f1:.4f}")

    # ── Immediately free GPU memory ──
    del m
    gc.collect()
    torch.cuda.empty_cache()

# ── Step 2: Rank and pick top-K ──
TOP_K_ENSEMBLE = 3   # adjust as needed
ranked = sorted(seed_scores.items(), key=lambda x: x[1], reverse=True)
print("\nAll seeds ranked:")
for s, f in ranked:
    print(f"  Seed {s}: {f:.4f}")

top_seeds = [s for s, _ in ranked[:TOP_K_ENSEMBLE]]
print(f"\nTop-{TOP_K_ENSEMBLE} seeds selected: {top_seeds}")

# ── Step 3: Load ONLY the top-K models for final ensemble ──
ensemble_models = []
for s in top_seeds:
    m = DebertaNER().to(device)
    m.load_state_dict(torch.load(seed_to_path[s], map_location=device))
    m.eval()
    ensemble_models.append(m)
    print(f"  Loaded seed {s} for ensemble ✅")

print(f"\nGPU memory after loading {TOP_K_ENSEMBLE} models:")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated")

In [ ]:
import torch, gc, os

seed_scores = {}

for s in SEEDS:
    path = seed_to_path[s]
    print(f"\nEvaluating seed {s}...")

    m = DebertaNER().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()

    # Check what evaluate() actually returns
    result = evaluate(m, val_loader, verbose=False)
    print(f"  Raw result type: {type(result)}, value: {result}")

    # Extract F1 — adjust based on what prints above
    if isinstance(result, tuple):
        f1 = result[0]   # try result[1] or result[2] if this is wrong
    elif isinstance(result, dict):
        f1 = result.get("f1") or result.get("f1_strict") or list(result.values())[0]
    else:
        f1 = float(result)

    seed_scores[s] = f1
    print(f"  Seed {s} → F1: {f1:.4f}")

    del m
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# from IPython.display import display

# # Unseen evaluation/submission source: DEV only
# eval_source_df = pd.read_csv(DEV_PATH).copy()

# id_candidates = ["ID", "id", "Id", "tweet_id", "uid"]
# id_col = next((c for c in id_candidates if c in eval_source_df.columns), None)
# if id_col is None:
#     id_col = "ID"
#     eval_source_df[id_col] = [f"dev_{i}" for i in range(len(eval_source_df))]


# def enforce_bio_consistency(tags: list) -> list:
#     fixed = []
#     prev_type = None
#     for t in tags:
#         if t == "O":
#             fixed.append("O"); prev_type = None; continue
#         if t.startswith("B-"):
#             ent = t[2:]; fixed.append(t); prev_type = ent; continue
#         if t.startswith("I-"):
#             ent = t[2:]
#             fixed.append(t if prev_type == ent else f"B-{ent}")
#             prev_type = ent; continue
#         fixed.append("O"); prev_type = None
#     return fixed


# @torch.no_grad()
# def predict_deberta_weighted_ensemble(models_with_weights: list, tokens: list) -> list:
#     if not tokens:
#         return []
#     enc = tokenizer(
#         tokens,
#         is_split_into_words=True,
#         padding="max_length",
#         truncation=True,
#         max_length=MAX_LEN,
#         return_tensors="pt",
#     )
#     logits_sum = None
#     input_ids      = enc["input_ids"].to(device)
#     attention_mask = enc["attention_mask"].to(device)
#     for item in models_with_weights:
#         out = item["model"](input_ids=input_ids, attention_mask=attention_mask)
#         # Apply per-seed temperature before weighting
#         T = item.get("temperature", 1.0)
#         weighted_logits = (out.logits / T) * item["weight"]
#         logits_sum = weighted_logits if logits_sum is None else logits_sum + weighted_logits
#     pred_ids = logits_sum.argmax(dim=-1).squeeze(0).tolist()
#     word_ids = enc.word_ids()
#     final, prev = [], None
#     for idx, w in enumerate(word_ids):
#         if w is not None and w != prev:
#             final.append(id2label[pred_ids[idx]])
#         prev = w
#     final = enforce_bio_consistency(final)
#     # ── First-person filter ──────────────────────────────────────────────────
#     final = apply_first_person_filter(tokens, final)
#     return final


# # Build DeBERTa-only ensemble from all seeds
# requested_seeds = [41, 44, 45, 46, 47, 49]
# seed_to_f1   = {r["seed"]: float(r.get("best_f1_calibrated", r["best_f1"])) for r in results} if "results" in globals() else {}
# seed_to_path = {r["seed"]: r["path"] for r in results} if "results" in globals() else {}
# seed_to_temp = {r["seed"]: float(r.get("temperature", 1.0)) for r in results} if "results" in globals() else {}

# missing = [s for s in requested_seeds if s not in seed_to_path]
# if missing:
#     raise RuntimeError(f"Missing DeBERTa checkpoints for seeds: {missing}. Run DeBERTa training first.")

# selected  = [{"seed": s, "path": seed_to_path[s], "f1": seed_to_f1.get(s, 1.0), "temperature": seed_to_temp.get(s, 1.0)} for s in requested_seeds]
# f1_sum    = sum(x["f1"] for x in selected) or float(len(selected))

# deberta_ensemble = []
# for item in selected:
#     m = load_best_model(item["path"])
#     deberta_ensemble.append({
#         "seed":        item["seed"],
#         "model":       m,
#         "weight":      item["f1"] / f1_sum,
#         "temperature": item["temperature"],
#     })

# print("Using DeBERTa-only ensemble (seeds 41,44,45,46,47,49) for unseen DEV eval/submission.")
# for m in deberta_ensemble:
#     print(f"  seed={m['seed']}  weight={m['weight']:.4f}  T={m['temperature']:.2f}")

# eval_rows = []
# for _, row in eval_source_df.iterrows():
#     tokens = parse_tokens(row["tokens"])
#     gold   = parse_labels(row["ner_tags"], len(tokens))
#     pred   = predict_deberta_weighted_ensemble(deberta_ensemble, tokens)
#     eval_rows.append({"ID": row[id_col], "tokens": tokens, "ner_tags_str": gold, "prediction": pred})

# eval_df = pd.DataFrame(eval_rows)

# strict_metrics = evaluate_test_strict_ner(
#     eval_df, gold_col="ner_tags_str", pred_col="prediction", print_report=False,
# )
# for metric_name, metric_value in strict_metrics.items():
#     print(f"{metric_name}: {metric_value:.4f}")

# ner_tags_str = eval_df["ner_tags_str"].apply(_to_list).tolist()
# prediction   = eval_df["prediction"].apply(_to_list).tolist()
# relaxed_results = calculate_f1_per_entity_covering_all(ner_tags_str, prediction)

# print("\nF1 Score Results Per Entity (Relaxed):")
# for entity, metrics in relaxed_results.items():
#     print(f"Entity Type: {entity}")
#     for metric, value in metrics.items():
#         print(f"  {metric}: {value}")
#     print()

# micro_f1           = strict_metrics["f1_strict"]
# overall_relaxed_f1 = relaxed_results["Overall"]["F1-Score"]

# print("=" * 50)
# print(f"Unseen DEV Micro F1 (Strict): {micro_f1:.4f}")
# print(f"Unseen DEV Relaxed F1:        {overall_relaxed_f1:.4f}")
# print("=" * 50)

# submission_df = eval_df[["ID", "prediction"]].rename(columns={"prediction": "predicted_ner_tags"}).copy()
# submission_df["predicted_ner_tags"] = submission_df["predicted_ner_tags"].apply(str)
# submission_path = "submission_dev_deberta_all_seeds.csv"
# submission_df.to_csv(submission_path, index=False)
# print(f"Saved submission: {submission_path}")
# display(submission_df.head(10))


In [ ]:
import torch, gc

gc.collect()                 # free Python memory
torch.cuda.empty_cache()     # release cached GPU memory
torch.cuda.ipc_collect()     # clean inter-process memory

In [ ]:
del model
del optimizer
del scheduler
del train_loader
del val_loader

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [26]:
from transformers import AutoTokenizer, AutoConfig
import os

# ── PubMedBERT constants (must match training) ────────────────────────────────
PUBMEDBERT_MODEL   = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
PUBMEDBERT_MAX_LEN = 512
PUBMEDBERT_SEEDS   = [41, 44]
PUBMEDBERT_PREFIX  = "best_pubmedbert_ner"

PUBMEDBERT_CFG = {
    "model_name": PUBMEDBERT_MODEL,
    "max_len":    PUBMEDBERT_MAX_LEN,
    "dropout":    0.1,
}

# ── Load tokenizer only (no training) ────────────────────────────────────────
pubmedbert_tokenizer = AutoTokenizer.from_pretrained(PUBMEDBERT_MODEL)
print(f"Tokenizer loaded: {PUBMEDBERT_MODEL}")
print(f"Vocab size: {pubmedbert_tokenizer.vocab_size}")

# ── Reconstruct checkpoint paths ─────────────────────────────────────────────
seed_to_path = {
    s: f"/kaggle/working/{PUBMEDBERT_PREFIX}_seed{s}.pt"
    for s in PUBMEDBERT_SEEDS
}

pubmedbert_results = [
    {"seed": s, "path": seed_to_path[s], "best_f1": 1.0}
    for s in PUBMEDBERT_SEEDS
    if os.path.exists(seed_to_path[s])
]

print(f"\nFound {len(pubmedbert_results)}/{len(PUBMEDBERT_SEEDS)} checkpoints:")
for r in pubmedbert_results:
    print(f"  Seed {r['seed']}: {r['path']} ✅")

missing = [s for s in PUBMEDBERT_SEEDS if not os.path.exists(seed_to_path[s])]
if missing:
    print(f"\n⚠️  Missing checkpoints for seeds: {missing}")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Vocab size: 30522

Found 2/2 checkpoints:
  Seed 41: /kaggle/working/best_pubmedbert_ner_seed41.pt ✅
  Seed 44: /kaggle/working/best_pubmedbert_ner_seed44.pt ✅


In [27]:
from IPython.display import display
import torch, gc

# ── Load unlabelled test set ──────────────────────────────────────────────────
test_df = pd.read_csv('/kaggle/input/datasets/shuvodey001/testdata/new_test_data (1).csv')
id_candidates = ["ID", "id", "Id", "tweet_id", "uid"]
id_col = next((c for c in id_candidates if c in test_df.columns), None)
if id_col is None:
    id_col = "ID"
    test_df[id_col] = [f"test_{i}" for i in range(len(test_df))]

all_tokens = [parse_tokens(row["tokens"]) for _, row in test_df.iterrows()]
all_ids    = list(test_df[id_col])
print(f"Test rows: {len(test_df)}")

# ── Build ensemble member list ────────────────────────────────────────────────
# 2 DeBERTa seeds + 2 PubMedBERT seeds, equal weight across all 4
deberta_paths     = {r["seed"]: r["path"] for r in results}
pubmedbert_paths  = {r["seed"]: r["path"] for r in pubmedbert_results}

ensemble_members = (
    [{"arch": "deberta",     "seed": s, "path": deberta_paths[s],    "weight": 1.0} for s in DEBERTA_SEEDS]
  + [{"arch": "pubmedbert",  "seed": s, "path": pubmedbert_paths[s], "weight": 1.0} for s in PUBMEDBERT_SEEDS]
)
total_weight = sum(m["weight"] for m in ensemble_members)
for m in ensemble_members:
    m["weight"] /= total_weight   # normalise to sum=1

print("Ensemble members:")
for m in ensemble_members:
    print(f"  {m['arch']:<12}  seed={m['seed']}  weight={m['weight']:.3f}  {m['path']}")

# ── Accumulate logits one model at a time ─────────────────────────────────────
accumulated_logits = None   # shape: (N_test, MAX_LEN_deberta, num_labels) aligned per-token

FIRST_PERSON_MARKERS = {
    "i", "i'm", "i've", "i'd", "i'll", "im", "ive", "id",
    "my", "me", "myself", "mine", "we", "we're", "our", "ours",
}

def _get_word_preds(logits_tensor, tokens, tok_obj, max_len):
    """Decode a (1, max_len, num_labels) logit tensor → word-level label list."""
    enc = tok_obj(
        tokens,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    word_ids  = enc.word_ids()
    pred_ids  = logits_tensor.argmax(dim=-1).squeeze(0).tolist()
    final, prev = [], None
    for idx, w in enumerate(word_ids):
        if w is not None and w != prev:
            final.append(id2label[pred_ids[idx]])
        prev = w
    return final

def _enforce_bio(tags):
    fixed, prev_type = [], None
    for t in tags:
        if t == "O":
            fixed.append("O"); prev_type = None; continue
        if t.startswith("B-"):
            ent = t[2:]; fixed.append(t); prev_type = ent; continue
        if t.startswith("I-"):
            ent = t[2:]
            fixed.append(t if prev_type == ent else f"B-{ent}"); prev_type = ent; continue
        fixed.append("O"); prev_type = None
    return fixed

def _fp_filter(tokens, pred_labels):
    if not any(t.lower() in FIRST_PERSON_MARKERS for t in tokens):
        return ["O"] * len(pred_labels)
    return pred_labels

# We run DeBERTa and PubMedBERT members separately because they have different
# max_len and tokenizers. We collect per-row WORD-level logit-equivalent scores
# by converting each model's word predictions into soft votes, then averaging.
#
# Strategy: for each model, get word-level argmax labels, convert to one-hot
# probability vectors, weight, and sum. Final prediction = argmax of summed votes.

N = len(all_tokens)
num_labels = len(label_list)
import numpy as np
vote_matrix = np.zeros((N, max(len(t) for t in all_tokens) + 1, num_labels), dtype=np.float32)

# Track per-row token lengths
token_lengths = [len(t) for t in all_tokens]

for member in ensemble_members:
    arch   = member["arch"]
    path   = member["path"]
    weight = member["weight"]
    tok_obj  = tokenizer if arch == "deberta" else pubmedbert_tokenizer
    max_len  = MAX_LEN   if arch == "deberta" else PUBMEDBERT_MAX_LEN

    print(f"\nLoading {arch} seed={member['seed']}  weight={weight:.3f}...")
    if arch == "deberta":
        m = DebertaNER().to(device)
    else:
        m = GenericNERModel(PUBMEDBERT_MODEL, PUBMEDBERT_CFG["dropout"]).to(device)
    m.load_state_dict(torch.load(path, map_location=device), strict=True)
    m.eval()

    with torch.no_grad():
        for row_idx, tokens in enumerate(all_tokens):
            if not tokens:
                continue
            enc = tok_obj(
                tokens,
                is_split_into_words=True,
                padding="max_length",
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            out = m(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
            )
            word_ids = enc.word_ids()
            logits   = out.logits.squeeze(0).cpu().numpy()  # (max_len, num_labels)
            # Map subtoken positions back to word positions
            prev_w = None
            for pos, w in enumerate(word_ids):
                if w is not None and w != prev_w and w < len(tokens):
                    # Soft vote: add weighted logit for this word position
                    vote_matrix[row_idx, w] += weight * logits[pos]
                prev_w = w

    del m
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Done. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Decode votes → final predictions ─────────────────────────────────────────
submission_rows = []

for row_idx, tokens in enumerate(all_tokens):
    n = len(tokens)
    if n == 0:
        submission_rows.append({id_col: all_ids[row_idx], "predicted_ner_tags": str(["O"])})
        continue

    word_votes = vote_matrix[row_idx, :n]           # (n_tokens, num_labels)
    pred_ids   = word_votes.argmax(axis=-1).tolist()
    final = [id2label[p] for p in pred_ids]
    final = _enforce_bio(final)
    final = _fp_filter(tokens, final)
    submission_rows.append({id_col: all_ids[row_idx], "predicted_ner_tags": str(final)})

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("/kaggle/working/submission_test_deberta_pubmedbert_ensemble.csv", index=False)
print(f"\nSaved {len(submission_df)} rows → submission_test_deberta_pubmedbert_ensemble.csv")
display(submission_df.head(10))


Test rows: 578
Ensemble members:
  deberta       seed=47  weight=0.250  /kaggle/working/best_deberta_ner_seed47.pt
  deberta       seed=49  weight=0.250  /kaggle/working/best_deberta_ner_seed49.pt
  pubmedbert    seed=41  weight=0.250  /kaggle/working/best_pubmedbert_ner_seed41.pt
  pubmedbert    seed=44  weight=0.250  /kaggle/working/best_pubmedbert_ner_seed44.pt

Loading deberta seed=47  weight=0.250...


Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

DebertaForTokenClassification LOAD REPORT from: microsoft/deberta-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Done. GPU: 0.01 GB

Loading deberta seed=49  weight=0.250...


Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

...

  Done. GPU: 0.01 GB

Loading pubmedbert seed=44  weight=0.250...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:


  Done. GPU: 0.01 GB

Saved 578 rows → submission_test_deberta_pubmedbert_ensemble.csv


,ID,predicted_ner_tags
0,test_147,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,test_270,"['O', 'O', 'O', 'O']"
2,test_213,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Clinica..."
3,test_175,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,test_128,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
5,test_101,"['O', 'B-ClinicalImpacts', 'O', 'O', 'O']"
6,test_24,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
7,test_143,"['O', 'O', 'B-ClinicalImpacts', 'O', 'O', 'O',..."
8,test_67,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
9,test_106,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [ ]:
# from IPython.display import display
# import torch, gc

# eval_source_df = pd.read_csv(DEV_PATH).copy()

# id_candidates = ["ID", "id", "Id", "tweet_id", "uid"]
# id_col = next((c for c in id_candidates if c in eval_source_df.columns), None)
# if id_col is None:
#     id_col = "ID"
#     eval_source_df[id_col] = [f"dev_{i}" for i in range(len(eval_source_df))]
# # requested_seeds = [49, 45, 46, 44, 47] best sf1 0.4590

# requested_seeds = [47, 49, 45, 46,41]
# seed_to_f1   = {r["seed"]: float(r.get("best_f1_calibrated", r["best_f1"])) for r in results} if "results" in globals() else {}
# seed_to_path = {r["seed"]: r["path"] for r in results} if "results" in globals() else {}
# seed_to_temp = {r["seed"]: float(r.get("temperature", 1.0)) for r in results} if "results" in globals() else {}

# missing = [s for s in requested_seeds if s not in seed_to_path]
# if missing:
#     raise RuntimeError(f"Missing DeBERTa checkpoints for seeds: {missing}.")

# selected  = [{"seed": s, "path": seed_to_path[s], "f1": seed_to_f1.get(s, 1.0), "temperature": seed_to_temp.get(s, 1.0)} for s in requested_seeds]
# f1_sum    = sum(x["f1"] for x in selected) or float(len(selected))
# for item in selected:
#     item["weight"] = item["f1"] / f1_sum

# # Pre-tokenize all rows once
# all_tokens = []
# all_golds  = []
# all_ids    = []
# for _, row in eval_source_df.iterrows():
#     tokens = parse_tokens(row["tokens"])
#     gold   = parse_labels(row["ner_tags"], len(tokens))
#     all_tokens.append(tokens)
#     all_golds.append(gold)
#     all_ids.append(row[id_col])

# # Accumulate temperature-scaled weighted logits — one model at a time
# accumulated_logits = None

# for item in selected:
#     T = item["temperature"]
#     w = item["weight"]
#     print(f"\nRunning seed {item['seed']}  T={T:.2f}  weight={w:.4f}...")
#     m = DebertaNER().to(device)
#     m.load_state_dict(torch.load(item["path"], map_location=device), strict=True)
#     m.eval()

#     model_logits = []
#     with torch.no_grad():
#         for tokens in all_tokens:
#             if not tokens:
#                 model_logits.append(torch.zeros(1, MAX_LEN, len(label_list)))
#                 continue
#             enc = tokenizer(
#                 tokens,
#                 is_split_into_words=True,
#                 padding="max_length",
#                 truncation=True,
#                 max_length=MAX_LEN,
#                 return_tensors="pt",
#             )
#             out = m(
#                 input_ids=enc["input_ids"].to(device),
#                 attention_mask=enc["attention_mask"].to(device),
#             )
#             model_logits.append((out.logits / T).cpu())   # ← temperature scaling

#     model_logits_tensor = torch.cat(model_logits, dim=0)
#     weighted = model_logits_tensor * w
#     accumulated_logits = weighted if accumulated_logits is None else accumulated_logits + weighted

#     del m, model_logits, model_logits_tensor, weighted
#     gc.collect()
#     torch.cuda.empty_cache()
#     print(f"  Done. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")


# def enforce_bio_consistency(tags):
#     fixed, prev_type = [], None
#     for t in tags:
#         if t == "O":
#             fixed.append("O"); prev_type = None
#         elif t.startswith("B-"):
#             fixed.append(t); prev_type = t[2:]
#         elif t.startswith("I-"):
#             ent = t[2:]
#             fixed.append(t if prev_type == ent else f"B-{ent}")
#             prev_type = ent
#         else:
#             fixed.append("O"); prev_type = None
#     return fixed


# pred_ids_all = accumulated_logits.argmax(dim=-1)

# eval_rows = []
# for i, (tokens, gold) in enumerate(zip(all_tokens, all_golds)):
#     if not tokens:
#         eval_rows.append({"ID": all_ids[i], "tokens": tokens, "ner_tags_str": gold, "prediction": []})
#         continue
#     enc = tokenizer(
#         tokens,
#         is_split_into_words=True,
#         padding="max_length",
#         truncation=True,
#         max_length=MAX_LEN,
#         return_tensors="pt",
#     )
#     word_ids = enc.word_ids()
#     pred_row = pred_ids_all[i].tolist()
#     final, prev = [], None
#     for idx, w in enumerate(word_ids):
#         if w is not None and w != prev:
#             final.append(id2label[pred_row[idx]])
#         prev = w
#     final = enforce_bio_consistency(final)
#     # ── First-person filter ──────────────────────────────────────────────────
#     final = apply_first_person_filter(tokens, final)
#     eval_rows.append({"ID": all_ids[i], "tokens": tokens, "ner_tags_str": gold, "prediction": final})

# eval_df = pd.DataFrame(eval_rows)

# strict_metrics = evaluate_test_strict_ner(eval_df, gold_col="ner_tags_str", pred_col="prediction", print_report=False)
# for metric_name, metric_value in strict_metrics.items():
#     print(f"{metric_name}: {metric_value:.4f}")

# ner_tags_str = eval_df["ner_tags_str"].apply(_to_list).tolist()
# prediction   = eval_df["prediction"].apply(_to_list).tolist()
# relaxed_results = calculate_f1_per_entity_covering_all(ner_tags_str, prediction)

# print("\nF1 Score Results Per Entity (Relaxed):")
# for entity, metrics in relaxed_results.items():
#     print(f"Entity Type: {entity}")
#     for metric, value in metrics.items():
#         print(f"  {metric}: {value}")
#     print()

# micro_f1           = strict_metrics["f1_strict"]
# overall_relaxed_f1 = relaxed_results["Overall"]["F1-Score"]
# print("=" * 50)
# print(f"Unseen DEV Micro F1 (Strict): {micro_f1:.4f}")
# print(f"Unseen DEV Relaxed F1:        {overall_relaxed_f1:.4f}")
# print("=" * 50)

# submission_df = eval_df[["ID", "prediction"]].rename(columns={"prediction": "predicted_ner_tags"}).copy()
# submission_df["predicted_ner_tags"] = submission_df["predicted_ner_tags"].apply(str)
# submission_path = "submission_dev_deberta_444546.csv"
# submission_df.to_csv(submission_path, index=False)
# print(f"Saved submission: {submission_path}")
# display(submission_df.head(10))


In [ ]:
import os
os.remove("/kaggle/working/state.db")

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated | "
              f"{torch.cuda.memory_reserved(i)/1e9:.2f} GB reserved")

In [ ]:
import torch, gc

# Kill all live tensors and variables
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            del obj
    except:
        pass

# Also delete common variable names if they exist
for var in ['model', 'optimizer', 'scheduler', 'train_loader', 'val_loader', 
            'inputs', 'outputs', 'loss', 'logits', 'embeddings']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated | "
              f"{torch.cuda.memory_reserved(i)/1e9:.2f} GB reserved")